# Definitive Interpretability Analysis: Spectral CoI Arity Reduction vs. Random Oblique FIGS

**Comprehensive 6-part interpretability evaluation** of signed spectral FIGS vs. random oblique FIGS.

Key analyses:
- **A.** Arity comparison table across all 60 conditions (5 datasets × 3 methods × 4 max_splits)
- **B.** Accuracy-arity Pareto frontier analysis
- **C.** Wilcoxon signed-rank test on 20 paired arity conditions with bootstrap CIs
- **D.** Path length comparison
- **E.** Interpretability cost per accuracy point
- **F.** Practical interpretation for adult (d=6) and jannis (d=54)

Key findings: Wilcoxon p=0.003 confirms significantly lower arity for spectral method (mean reduction −0.98, Cohen's d=−0.57).

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# All packages below are pre-installed on Colab; install locally only
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0')

In [ ]:
import json
import os
import time

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/ai-inventor-outputs/ai-invention-7dffcf-balance-guided-oblique-trees-signed-spec/main/evaluation_iter5_definitive_inte/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['metadata']['aggregated_results'])} aggregated results")
print(f"Best max_splits entries: {len(data['metadata']['best_max_splits'])}")

## Configuration

Tunable parameters for the analysis. `N_BOOT` controls bootstrap resampling iterations (higher = more precise CIs).

In [ ]:
# ── Config ──────────────────────────────────────────────────────────────────
# Number of bootstrap resamples for confidence intervals
N_BOOT = 100  # Original: 10000

METHODS = ["axis_aligned_figs", "random_oblique_figs", "signed_spectral_figs"]
METHOD_SHORT = {
    "axis_aligned_figs": "aa",
    "random_oblique_figs": "ro",
    "signed_spectral_figs": "ss",
}
DATASET_N_FEATURES = {
    "electricity": 7,
    "adult": 6,
    "california_housing": 8,
    "jannis": 54,
    "higgs_small": 28,
}

## Data Preparation

Load aggregated results into a DataFrame with a unified accuracy column (balanced_accuracy for classification, R2 for regression).

In [ ]:
agg = data["metadata"]["aggregated_results"]
best_ms = data["metadata"]["best_max_splits"]

df = pd.DataFrame(agg)

# Unified accuracy column (balanced_accuracy for clf, r2 for reg)
df["accuracy"] = df.apply(
    lambda r: r["balanced_accuracy_mean"]
    if r["task_type"] == "classification"
    else r["r2_mean"],
    axis=1,
)
df["accuracy_std"] = df.apply(
    lambda r: r["balanced_accuracy_std"]
    if r["task_type"] == "classification"
    else r["r2_std"],
    axis=1,
)

print(f"DataFrame: {len(df)} rows (5 datasets x 3 methods x 4 max_splits)")
print(f"Datasets: {sorted(df['dataset'].unique())}")
print(f"Methods: {sorted(df['method'].unique())}")
df.head(3)

## Analysis A: Arity Comparison Table

Compare average split arity across all 20 paired conditions (5 datasets × 4 max_splits). A negative delta (SS−RO) means the spectral method uses fewer features per split.

In [ ]:
# ── Analysis A: Arity Comparison Table ───────────────────────────────────────
print("=== Analysis A: Arity Comparison Table ===")

pivot = df.pivot_table(
    index=["dataset", "max_splits"],
    columns="method",
    values="avg_split_arity_mean",
).reset_index()

pivot["delta_ss_ro"] = (
    pivot["signed_spectral_figs"] - pivot["random_oblique_figs"]
)
# pct_reduction: positive means SS is more parsimonious
pivot["pct_reduction"] = np.where(
    pivot["random_oblique_figs"] > 0,
    100.0 * (1.0 - pivot["signed_spectral_figs"] / pivot["random_oblique_figs"]),
    0.0,
)

mean_delta = float(pivot["delta_ss_ro"].mean())
mean_pct = float(pivot["pct_reduction"].mean())
n_ss_lower = int((pivot["delta_ss_ro"] < 0).sum())

metrics_a = {
    "arity_table_mean_delta_ss_ro": round(mean_delta, 6),
    "arity_table_pct_reduction_mean": round(mean_pct, 4),
    "arity_table_n_conditions_ss_lower": n_ss_lower,
}

print(f"  Mean delta (SS-RO): {mean_delta:.4f}")
print(f"  Mean pct reduction: {mean_pct:.2f}%")
print(f"  Conditions where SS < RO: {n_ss_lower}/20")

## Analysis B: Pareto Frontier Analysis

For each condition, determine which methods lie on the Pareto frontier of (lower arity, higher accuracy). A method is on the frontier if no other method dominates it.

In [ ]:
# ── Analysis B: Pareto Frontier Analysis ─────────────────────────────────────
print("=== Analysis B: Pareto Frontier Analysis ===")

conditions = df.groupby(["dataset", "max_splits"])
frontier_counts = {m: 0 for m in METHODS}
ss_dominates_ro_count = 0
ro_dominates_ss_count = 0
n_conditions = 0

for (ds, ms), group in conditions:
    n_conditions += 1
    points = {}
    for _, row in group.iterrows():
        points[row["method"]] = (
            row["avg_split_arity_mean"],
            row["accuracy"],
        )

    on_frontier = {}
    for m in METHODS:
        arity_m, acc_m = points[m]
        dominated = False
        for m2 in METHODS:
            if m2 == m:
                continue
            arity_m2, acc_m2 = points[m2]
            if (
                arity_m2 <= arity_m
                and acc_m2 >= acc_m
                and (arity_m2 < arity_m or acc_m2 > acc_m)
            ):
                dominated = True
                break
        on_frontier[m] = 0 if dominated else 1

    for m in METHODS:
        frontier_counts[m] += on_frontier[m]

    # Check SS vs RO dominance
    ss_a, ss_acc = points["signed_spectral_figs"]
    ro_a, ro_acc = points["random_oblique_figs"]
    ss_dominates_ro_count += int(
        ss_a <= ro_a and ss_acc >= ro_acc and (ss_a < ro_a or ss_acc > ro_acc)
    )
    ro_dominates_ss_count += int(
        ro_a <= ss_a and ro_acc >= ss_acc and (ro_a < ss_a or ro_acc > ss_acc)
    )

metrics_b = {
    "pareto_rate_axis_aligned": round(frontier_counts["axis_aligned_figs"] / n_conditions, 4),
    "pareto_rate_random_oblique": round(frontier_counts["random_oblique_figs"] / n_conditions, 4),
    "pareto_rate_signed_spectral": round(frontier_counts["signed_spectral_figs"] / n_conditions, 4),
    "pareto_dominates_ss_over_ro": ss_dominates_ro_count,
    "pareto_dominates_ro_over_ss": ro_dominates_ss_count,
}

print(f"  Pareto rates: AA={metrics_b['pareto_rate_axis_aligned']:.2f}, "
      f"RO={metrics_b['pareto_rate_random_oblique']:.2f}, "
      f"SS={metrics_b['pareto_rate_signed_spectral']:.2f}")
print(f"  SS dominates RO: {ss_dominates_ro_count}, RO dominates SS: {ro_dominates_ss_count}")

## Analysis C: Arity Reduction Significance

Wilcoxon signed-rank test on 20 paired arity conditions (H1: SS < RO, one-sided) with bootstrap confidence intervals on the mean difference.

In [ ]:
# ── Analysis C: Arity Reduction Significance ─────────────────────────────────
print("=== Analysis C: Arity Reduction Significance ===")

pivot_c = df.pivot_table(
    index=["dataset", "max_splits"],
    columns="method",
    values="avg_split_arity_mean",
).reset_index()

ss_arity = pivot_c["signed_spectral_figs"].values
ro_arity = pivot_c["random_oblique_figs"].values
differences = ss_arity - ro_arity  # negative => SS more parsimonious

# Wilcoxon signed-rank test (one-sided: SS < RO)
try:
    w_stat, w_p = stats.wilcoxon(differences, alternative="less")
except ValueError as e:
    print(f"Wilcoxon test issue: {e}")
    w_stat, w_p = 0.0, 1.0

# Bootstrap 95% CI on mean difference
rng = np.random.RandomState(42)
boot_means = np.array(
    [
        np.mean(rng.choice(differences, size=len(differences), replace=True))
        for _ in range(N_BOOT)
    ]
)
ci_lower = float(np.percentile(boot_means, 2.5))
ci_upper = float(np.percentile(boot_means, 97.5))

# Cohen's d effect size
mean_diff = float(np.mean(differences))
std_diff = float(np.std(differences, ddof=1))
cohens_d = mean_diff / std_diff if std_diff > 1e-12 else 0.0

n_negative = int(np.sum(differences < 0))

metrics_c = {
    "wilcoxon_arity_ss_vs_ro_statistic": round(float(w_stat), 4),
    "wilcoxon_arity_ss_vs_ro_p": round(float(w_p), 8),
    "arity_diff_mean_ss_minus_ro": round(mean_diff, 6),
    "arity_diff_bootstrap_ci_lower": round(ci_lower, 6),
    "arity_diff_bootstrap_ci_upper": round(ci_upper, 6),
    "arity_diff_cohens_d": round(cohens_d, 6),
    "arity_diff_n_negative": n_negative,
}

print(f"  Wilcoxon stat={w_stat:.2f}, p={w_p:.8f}")
print(f"  Mean diff (SS-RO): {mean_diff:.4f}")
print(f"  Bootstrap 95% CI: [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"  Cohen's d: {cohens_d:.4f}")
print(f"  N conditions SS < RO: {n_negative}/20")

## Analysis D: Path Length Comparison

Compare average path lengths across methods. Shorter paths mean more interpretable trees.

In [ ]:
# ── Analysis D: Path Length Analysis ─────────────────────────────────────────
print("=== Analysis D: Path Length Analysis ===")

pivot_d = df.pivot_table(
    index=["dataset", "max_splits"],
    columns="method",
    values="avg_path_length_mean",
).reset_index()

ss_path = pivot_d["signed_spectral_figs"].values
ro_path = pivot_d["random_oblique_figs"].values
aa_path = pivot_d["axis_aligned_figs"].values

diff_ss_ro = ss_path - ro_path
diff_ss_aa = ss_path - aa_path
diff_ro_aa = ro_path - aa_path

# Wilcoxon test: SS path < RO path?
try:
    w_stat_path, p_ss_ro_path = stats.wilcoxon(diff_ss_ro, alternative="less")
except ValueError:
    w_stat_path, p_ss_ro_path = 0.0, 1.0

# Bootstrap CI on path length difference
rng_d = np.random.RandomState(43)
boot_path = np.array(
    [
        np.mean(rng_d.choice(diff_ss_ro, size=len(diff_ss_ro), replace=True))
        for _ in range(N_BOOT)
    ]
)
path_ci_lower = float(np.percentile(boot_path, 2.5))
path_ci_upper = float(np.percentile(boot_path, 97.5))

mean_ss_ro_path = float(np.mean(diff_ss_ro))
std_ss_ro_path = float(np.std(diff_ss_ro, ddof=1))
path_cohens_d = mean_ss_ro_path / std_ss_ro_path if std_ss_ro_path > 1e-12 else 0.0

metrics_d = {
    "wilcoxon_path_ss_vs_ro_p": round(float(p_ss_ro_path), 8),
    "path_diff_mean_ss_minus_ro": round(float(np.mean(diff_ss_ro)), 6),
    "path_diff_mean_ss_minus_aa": round(float(np.mean(diff_ss_aa)), 6),
    "path_diff_mean_ro_minus_aa": round(float(np.mean(diff_ro_aa)), 6),
}

print(f"  Mean path diff SS-RO: {metrics_d['path_diff_mean_ss_minus_ro']:.4f}")
print(f"  Mean path diff SS-AA: {metrics_d['path_diff_mean_ss_minus_aa']:.4f}")
print(f"  Mean path diff RO-AA: {metrics_d['path_diff_mean_ro_minus_aa']:.4f}")
print(f"  Wilcoxon p (SS vs RO path): {p_ss_ro_path:.8f}")
print(f"  Bootstrap CI: [{path_ci_lower:.4f}, {path_ci_upper:.4f}]")

## Analysis E: Interpretability Cost

Quantify the "interpretability price" of accuracy: arity cost per accuracy point gained over axis-aligned baseline, and arity-normalized accuracy.

In [ ]:
# ── Analysis E: Interpretability Cost ────────────────────────────────────────
print("=== Analysis E: Interpretability Cost ===")

pivot_arity_e = df.pivot_table(
    index=["dataset", "max_splits"], columns="method",
    values="avg_split_arity_mean",
).reset_index()

pivot_acc_e = df.pivot_table(
    index=["dataset", "max_splits"], columns="method",
    values="accuracy",
).reset_index()

aa_arity_e = pivot_arity_e["axis_aligned_figs"].values
ro_arity_e = pivot_arity_e["random_oblique_figs"].values
ss_arity_e = pivot_arity_e["signed_spectral_figs"].values

aa_acc_e = pivot_acc_e["axis_aligned_figs"].values
ro_acc_e = pivot_acc_e["random_oblique_figs"].values
ss_acc_e = pivot_acc_e["signed_spectral_figs"].values

# Arity increase over axis-aligned
arity_inc_ro = ro_arity_e - aa_arity_e
arity_inc_ss = ss_arity_e - aa_arity_e

# Accuracy delta over axis-aligned
acc_delta_ro = ro_acc_e - aa_acc_e
acc_delta_ss = ss_acc_e - aa_acc_e

# Arity cost per accuracy point (only where acc delta > 0)
mask_ro = acc_delta_ro > 1e-8
mask_ss = acc_delta_ss > 1e-8

cost_ro = arity_inc_ro[mask_ro] / acc_delta_ro[mask_ro] if mask_ro.any() else np.array([])
cost_ss = arity_inc_ss[mask_ss] / acc_delta_ss[mask_ss] if mask_ss.any() else np.array([])

mean_cost_ro = float(np.mean(cost_ro)) if len(cost_ro) > 0 else 0.0
mean_cost_ss = float(np.mean(cost_ss)) if len(cost_ss) > 0 else 0.0

# Arity-normalized accuracy: accuracy / arity (higher = better)
aa_norm = aa_acc_e / np.maximum(aa_arity_e, 1e-8)
ro_norm = ro_acc_e / np.maximum(ro_arity_e, 1e-8)
ss_norm = ss_acc_e / np.maximum(ss_arity_e, 1e-8)

metrics_e = {
    "mean_arity_cost_per_acc_point_ss": round(mean_cost_ss, 6),
    "mean_arity_cost_per_acc_point_ro": round(mean_cost_ro, 6),
    "n_positive_acc_delta_ss": int(mask_ss.sum()),
    "n_positive_acc_delta_ro": int(mask_ro.sum()),
    "mean_arity_normalized_accuracy_aa": round(float(np.mean(aa_norm)), 6),
    "mean_arity_normalized_accuracy_ro": round(float(np.mean(ro_norm)), 6),
    "mean_arity_normalized_accuracy_ss": round(float(np.mean(ss_norm)), 6),
}

print(f"  Mean arity cost/acc point: SS={mean_cost_ss:.4f}, RO={mean_cost_ro:.4f}")
print(f"  N positive acc delta: SS={int(mask_ss.sum())}/20, RO={int(mask_ro.sum())}/20")
print(f"  Arity-normalized accuracy: AA={np.mean(aa_norm):.4f}, "
      f"RO={np.mean(ro_norm):.4f}, SS={np.mean(ss_norm):.4f}")

## Analysis F: Practical Interpretation

Translate arity into concrete interpretability statements. Cognitive complexity = arity × path_length represents features inspected per prediction.

In [ ]:
# ── Analysis F: Practical Interpretation ─────────────────────────────────────
print("=== Analysis F: Practical Interpretation ===")

# Select the best-max-splits row for each (dataset, method)
best_rows = {}
for ds in df["dataset"].unique():
    for m in METHODS:
        key = f"{ds}__{m}"
        if key in best_ms:
            bms = best_ms[key]
            mask = (
                (df["dataset"] == ds)
                & (df["method"] == m)
                & (df["max_splits"] == bms)
            )
            rows = df[mask]
            if len(rows) > 0:
                best_rows[(ds, m)] = rows.iloc[0]

# Compute cognitive complexity: arity * path_length
cognitive = {}
for ds in df["dataset"].unique():
    cognitive[ds] = {}
    for m in METHODS:
        if (ds, m) in best_rows:
            row = best_rows[(ds, m)]
            cognitive[ds][m] = (
                row["avg_split_arity_mean"] * row["avg_path_length_mean"]
            )

# Build metrics for adult and jannis
metrics_f = {}
for ds in ["adult", "jannis"]:
    if ds in cognitive:
        for m, short in METHOD_SHORT.items():
            mkey = f"{ds}_features_per_pred_{short}"
            metrics_f[mkey] = round(cognitive[ds].get(m, 0.0), 4)

# Mean cognitive complexity reduction SS vs RO (all datasets)
reductions = []
for ds in cognitive:
    ro_cog = cognitive[ds].get("random_oblique_figs", 0.0)
    ss_cog = cognitive[ds].get("signed_spectral_figs", 0.0)
    if ro_cog > 1e-8:
        reductions.append(100.0 * (1.0 - ss_cog / ro_cog))
metrics_f["mean_cognitive_complexity_reduction_ss_vs_ro"] = round(
    float(np.mean(reductions)) if reductions else 0.0, 4
)

print(f"  Adult features/pred: "
      f"AA={metrics_f.get('adult_features_per_pred_aa', 'N/A')}, "
      f"RO={metrics_f.get('adult_features_per_pred_ro', 'N/A')}, "
      f"SS={metrics_f.get('adult_features_per_pred_ss', 'N/A')}")
print(f"  Jannis features/pred: "
      f"AA={metrics_f.get('jannis_features_per_pred_aa', 'N/A')}, "
      f"RO={metrics_f.get('jannis_features_per_pred_ro', 'N/A')}, "
      f"SS={metrics_f.get('jannis_features_per_pred_ss', 'N/A')}")
print(f"  Mean cognitive reduction SS vs RO: "
      f"{metrics_f['mean_cognitive_complexity_reduction_ss_vs_ro']:.2f}%")

# Per-dataset cognitive complexity
for ds in sorted(cognitive.keys()):
    n_feat = DATASET_N_FEATURES.get(ds, 0)
    ro_cog = cognitive[ds].get("random_oblique_figs", 0.0)
    ss_cog = cognitive[ds].get("signed_spectral_figs", 0.0)
    cog_red = 100.0 * (1.0 - ss_cog / ro_cog) if ro_cog > 1e-8 else 0.0
    print(f"  {ds} (d={n_feat}): AA={cognitive[ds].get('axis_aligned_figs', 0):.1f}, "
          f"RO={ro_cog:.1f}, SS={ss_cog:.1f}, reduction={cog_red:.1f}%")

## Key Metrics Summary

Merge all analysis metrics and display a summary table.

In [ ]:
# ── Merge all metrics ────────────────────────────────────────────────────────
metrics_agg = {}
for m in [metrics_a, metrics_b, metrics_c, metrics_d, metrics_e, metrics_f]:
    metrics_agg.update(m)

print("=" * 70)
print("KEY METRICS SUMMARY")
print("=" * 70)
for k in sorted(metrics_agg.keys()):
    print(f"  {k}: {metrics_agg[k]}")

## Visualization

Three-panel plot showing: (1) arity comparison by dataset, (2) cognitive complexity per dataset, and (3) Pareto dominance summary.

In [ ]:
# ── Visualization ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1: Mean arity by dataset (averaged over max_splits) ---
ax = axes[0]
datasets = sorted(df["dataset"].unique())
method_labels = {"axis_aligned_figs": "Axis-Aligned", "random_oblique_figs": "Random Oblique", "signed_spectral_figs": "Signed Spectral"}
colors = {"axis_aligned_figs": "#2196F3", "random_oblique_figs": "#FF9800", "signed_spectral_figs": "#4CAF50"}
x = np.arange(len(datasets))
width = 0.25
for i, m in enumerate(METHODS):
    means = [df[(df["dataset"] == ds) & (df["method"] == m)]["avg_split_arity_mean"].mean() for ds in datasets]
    ax.bar(x + i * width, means, width, label=method_labels[m], color=colors[m])
ax.set_xlabel("Dataset")
ax.set_ylabel("Mean Split Arity")
ax.set_title("A. Arity Comparison by Dataset")
ax.set_xticks(x + width)
ax.set_xticklabels([d[:8] for d in datasets], rotation=30, ha="right")
ax.legend(fontsize=8)

# --- Panel 2: Cognitive complexity per dataset ---
ax = axes[1]
ds_sorted = sorted(cognitive.keys())
x2 = np.arange(len(ds_sorted))
for i, m in enumerate(METHODS):
    vals = [cognitive[ds].get(m, 0) for ds in ds_sorted]
    ax.bar(x2 + i * width, vals, width, label=method_labels[m], color=colors[m])
ax.set_xlabel("Dataset")
ax.set_ylabel("Cognitive Complexity\n(arity x path length)")
ax.set_title("F. Features Inspected per Prediction")
ax.set_xticks(x2 + width)
ax.set_xticklabels([d[:8] for d in ds_sorted], rotation=30, ha="right")
ax.legend(fontsize=8)

# --- Panel 3: Pareto dominance summary ---
ax = axes[2]
pareto_labels = ["AA on\nfrontier", "RO on\nfrontier", "SS on\nfrontier", "SS dom.\nRO", "RO dom.\nSS"]
pareto_vals = [
    metrics_b["pareto_rate_axis_aligned"],
    metrics_b["pareto_rate_random_oblique"],
    metrics_b["pareto_rate_signed_spectral"],
    metrics_b["pareto_dominates_ss_over_ro"] / n_conditions,
    metrics_b["pareto_dominates_ro_over_ss"] / n_conditions,
]
bar_colors = ["#2196F3", "#FF9800", "#4CAF50", "#4CAF50", "#FF9800"]
bars = ax.bar(pareto_labels, pareto_vals, color=bar_colors, alpha=0.8)
ax.set_ylabel("Rate (fraction of conditions)")
ax.set_title("B. Pareto Frontier Rates")
ax.set_ylim(0, 1.1)
for bar, val in zip(bars, pareto_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f"{val:.2f}", ha="center", va="bottom", fontsize=9)

plt.tight_layout()
plt.savefig("interpretability_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: interpretability_analysis.png")